In [1]:
import json
import re
import pandas as pd
from typing import Dict
import html 
bacteria_regex = r'(?P<bacterium>[A-Z]\. [a-z]+)(?P<strain>\s?[A-Z]+\s?[0-9]+)?'

In [2]:
df = pd.read_csv('APD6_processed.csv',dtype={"APD_ID":str})
print(f"初始数据集中存在的行总共有{len(df)}\n")
df["APD_ID"] = "AP" + df["APD_ID"]
# 氨基酸序列要清洗，保留20种标准的氨基酸字母
def clean_sequence(seq: str) -> str:
    seq = seq.upper().strip()
    standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if set(seq).issubset(standard_aa):
        return seq
    else:
        return None
df["Sequence"] = df["Sequence"].apply(clean_sequence)
df = df.dropna(subset=['Sequence'])
print(f"氨基酸序列清洗后存在的行总共有{len(df)}\n")
df.head(5)

/tmp/ipykernel_2783122/2703227181.py:1: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,26

初始数据集中存在的行总共有6070

氨基酸序列清洗后存在的行总共有6062



,APD_ID,Sequence,Length,Activity,MIC (A. acidoterrestris ATCC 49025),MIC (A. acidoterrestris),MIC (A. actinomycetemcomitans),MIC (A. alternata),MIC (A. alternatam),MIC (A. aumannii B28),...,HC50 ( female murine RBC),HC50 ( highly hemolytic),HC50 ( horse RBC),HC50 ( human RBC),HC50 ( moderate hemolytic),HC50 ( murine RBC),HC50 ( porcine RBC),HC50 ( rat RBC),HC50 (In vitro Toxicity),HC50 (lytic)
0,AP00004,NLCERASLTWTGNCGNTGHCDTQCRNWESAKHGACHKRGNWKCFCYFDC,49,"Anti-Gram+, Antifungal",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AP00001,GLWSKIKEVGKEAAKAAAKAAGKAALGAVSEAV,33,"Anti-Gram+ & Gram-, Antifungal, candidacidal, ...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AP00002,YVPLPNVPQPGRRPFPTFPGQGPFNPKIKWPQGY,34,Anti-Gram+ & Gram-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AP00003,DGVKLCDVPSGTWSGHCGSSSKCSQQCKDREHFAYGGACHYQFPSV...,54,Antifungal,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AP00006,GNNRPVYIPQPRPPHPRI,18,"Anti-Gram+ & Gram-, anti-sepsis",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
id_vars = ['APD_ID', 'Sequence', 'Length', 'Activity']

# 3. 自动筛选所有以 "HC50" 开头的列作为转换的目标
value_vars = [col for col in df.columns if col.startswith('HC50')]
long_df = df.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Raw_Species',
    value_name='HC50_Value'
)
long_df.head(2)

,APD_ID,Sequence,Length,Activity,Raw_Species,HC50_Value
0,AP00004,NLCERASLTWTGNCGNTGHCDTQCRNWESAKHGACHKRGNWKCFCYFDC,49,"Anti-Gram+, Antifungal",HC50 ( Human RBC),NaN
1,AP00001,GLWSKIKEVGKEAAKAAAKAAGKAALGAVSEAV,33,"Anti-Gram+ & Gram-, Antifungal, candidacidal, ...",HC50 ( Human RBC),NaN


In [4]:
long_df.dropna(subset=['HC50_Value'], inplace=True)
print(f"清洗后存在HC50值的行总共有{len(long_df)}\n")
long_df.head(2)

清洗后存在HC50值的行总共有178



,APD_ID,Sequence,Length,Activity,Raw_Species,HC50_Value
344,AP00367,GGLRSLGRKILRAWKKYGPIIVPIIRIG,28,"Anti-Gram+ & Gram-, Antiviral, Antifungal, can...",HC50 ( Human RBC),<100 uM
9152,AP03328,TLKIIVKVVKYLV,13,"Anti-Gram+ & Gram-, Anti-MRSA, Hemolytic",HC50 ( highly hemolytic),=25.1 ug/ml


In [5]:
long_df['Raw_Species'].value_counts()

Raw_Species
HC50 ( porcine RBC)           65
HC50 ( human RBC)             49
HC50 ( Human RBC)             19
HC50 ( murine RBC)            13
HC50 ( rat RBC)               10
HC50 ( horse RBC)              7
HC50 ( Hemolysis)              4
HC50 (  human RBC)             4
HC50 ( female murine RBC)      1
HC50 (  Human RBC)             1
HC50 (  highly hemolytic)      1
HC50 ( highly hemolytic)       1
HC50 ( moderate hemolytic)     1
HC50 (In vitro Toxicity)       1
HC50 (lytic)                   1
Name: count, dtype: int64

In [6]:
long_df['HC50_Value'].value_counts()


HC50_Value
>128 ug/ml     66
>320 uM        13
>250 uM        13
>128 uM         8
>250 ug/ml      8
>500 uM         6
=>128 ug/ml     6
>686 uM         5
>100 uM         4
>256 uM         4
>500 ug/ml      3
>>200 uM        3
>256 ug/ml      3
>265 uM         2
>100 ug/ml      2
=32 uM          2
>>250 ug/ml     2
>200 uM         2
<100 uM         2
=50 uM          1
=6.3 uM         1
=5 0            1
=25 uM          1
=25.1 ug/ml     1
>150 ug/ml      1
>1280 ug/ml     1
=3 2            1
>512 ug/ml      1
>150 uM         1
>300 uM         1
>125 ug/ml      1
>110.6 uM       1
=1 6            1
=18.8 ug/ml     1
<10 ug/ml       1
>381 ug/ml      1
=41.87 uM       1
>600 uM         1
>1000 ug/ml     1
=73.1 ug/ml     1
>122.1 uM       1
>230.6 uM       1
>>50 uM         1
Name: count, dtype: int64

In [8]:

df = pd.read_csv('APD6_processed.csv',dtype={"APD_ID":str})
print(f"初始数据集中存在的行总共有{len(df)}\n")
df["APD_ID"] = "AP" + df["APD_ID"]
# 氨基酸序列要清洗，保留20种标准的氨基酸字母
def clean_sequence(seq: str) -> str:
    seq = seq.upper().strip()
    standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if set(seq).issubset(standard_aa):
        return seq
    else:
        return None
df["Sequence"] = df["Sequence"].apply(clean_sequence)
df = df.dropna(subset=['Sequence'])
print(f"氨基酸序列清洗后存在的行总共有{len(df)}\n")
df.head(5)

/tmp/ipykernel_1603596/2566429194.py:1: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,26

初始数据集中存在的行总共有6070

氨基酸序列清洗后存在的行总共有6062



,APD_ID,Sequence,Length,Activity,MIC (A. acidoterrestris ATCC 49025),MIC (A. acidoterrestris),MIC (A. actinomycetemcomitans),MIC (A. alternata),MIC (A. alternatam),MIC (A. aumannii B28),...,HC50 ( female murine RBC),HC50 ( highly hemolytic),HC50 ( horse RBC),HC50 ( human RBC),HC50 ( moderate hemolytic),HC50 ( murine RBC),HC50 ( porcine RBC),HC50 ( rat RBC),HC50 (In vitro Toxicity),HC50 (lytic)
0,AP00004,NLCERASLTWTGNCGNTGHCDTQCRNWESAKHGACHKRGNWKCFCYFDC,49,"Anti-Gram+, Antifungal",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AP00001,GLWSKIKEVGKEAAKAAAKAAGKAALGAVSEAV,33,"Anti-Gram+ & Gram-, Antifungal, candidacidal, ...",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AP00002,YVPLPNVPQPGRRPFPTFPGQGPFNPKIKWPQGY,34,Anti-Gram+ & Gram-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AP00003,DGVKLCDVPSGTWSGHCGSSSKCSQQCKDREHFAYGGACHYQFPSV...,54,Antifungal,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AP00006,GNNRPVYIPQPRPPHPRI,18,"Anti-Gram+ & Gram-, anti-sepsis",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 做一个demo csv

id_vars = ['APD_ID', 'Sequence', 'Length', 'Activity']

# 3. 自动筛选所有以 "MIC(E. coli)" 开头的列作为转换的目标
value_vars = [col for col in df.columns if col.startswith('MIC(E. coli')]
demo_df = df.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Raw_Species',
    value_name='MIC_Value'
)
demo_df = demo_df.dropna(subset=['MIC_Value'])
print(f"清洗后存在MIC值的行总共有{len(demo_df)}\n")
demo_df.head(2)

In [7]:

id_vars = ['APD_ID', 'Sequence', 'Length', 'Activity']

# 3. 自动筛选所有以 "MIC" 开头的列作为转换的目标
value_vars = [col for col in df.columns if col.startswith('MIC')]
long_df = df.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Raw_Species',
    value_name='MIC_Value'
)
long_df.head(2)

,APD_ID,Sequence,Length,Activity,Raw_Species,MIC_Value
0,AP00004,NLCERASLTWTGNCGNTGHCDTQCRNWESAKHGACHKRGNWKCFCYFDC,49,"Anti-Gram+, Antifungal",MIC (A. acidoterrestris ATCC 49025),NaN
1,AP00001,GLWSKIKEVGKEAAKAAAKAAGKAALGAVSEAV,33,"Anti-Gram+ & Gram-, Antifungal, candidacidal, ...",MIC (A. acidoterrestris ATCC 49025),NaN


In [ ]:
long_df["MIC_Value"].describe()

count      16697
unique      1362
top       8  uM)
freq         523
Name: MIC_Value, dtype: object

In [8]:


long_df = long_df.dropna(subset=['MIC_Value'])
print(f"MIC 值存在的行总共有{len(long_df)}\n")

# MIC值变成数值，加上单位，统一处理成μg/mL

from Bio.SeqUtils import molecular_weight
def unify_to_ug_ml(row):
    # mic值是几何平均，没有大于或者小于
    raw_val = str(row['MIC_Value']).lower()
    seq = row['Sequence']
    
    # 1. 提取数值
    match = re.search(r'([\d\.]+)', raw_val)
    if not match: return None
    value = float(match.group(1))

    
    # 2. 如果已经是 ug/ml，直接返回数值
    if 'ug/ml' in raw_val:
        return value
        
    # 3. 如果是 uM，则根据序列计算分子量并转换
    if 'um' in raw_val:
        # 计算分子量 (Da)
        mw = molecular_weight(seq, seq_type="protein")
        print(f"分子质量是{mw} g/mol\n")
        # 转换公式：(uM * MW) / 1000 = ug/ml
        return (value * mw) / 1000
    
    return value # 默认返回
long_df["MIC_Value(ug/ml)"] = long_df.apply(unify_to_ug_ml,axis=1)
long_df = long_df.dropna(subset=['MIC_Value(ug/ml)'])
print(f"MIC 值清洗后存在的行总共有{len(long_df)}\n")
# 16695


MIC 值存在的行总共有16697

分子质量是1044.2255 g/mol

分子质量是2549.041 g/mol

分子质量是4614.266900000001 g/mol

分子质量是2902.3104000000003 g/mol

分子质量是3178.7704000000003 g/mol

分子质量是3164.7438 g/mol

分子质量是3148.7444000000005 g/mol

分子质量是3252.8456 g/mol

分子质量是1516.9088000000002 g/mol

分子质量是1550.9250000000002 g/mol

分子质量是607.4789000000001 g/mol

分子质量是2142.6029999999996 g/mol

分子质量是2176.6191999999996 g/mol

分子质量是2326.7954999999997 g/mol

分子质量是2540.0325 g/mol

分子质量是2171.6443 g/mol

分子质量是2191.6735999999996 g/mol

分子质量是4872.453500000002 g/mol

分子质量是1091.3435 g/mol

分子质量是2383.8468 g/mol

分子质量是2682.2243 g/mol

分子质量是2592.2161 g/mol

分子质量是2760.2964999999995 g/mol

分子质量是2696.2576999999997 g/mol

分子质量是3164.7239 g/mol

分子质量是2932.3923000000004 g/mol

分子质量是2589.9983000000007 g/mol

分子质量是1883.2978000000003 g/mol

分子质量是1939.4041000000002 g/mol

分子质量是6224.417800000003 g/mol

分子质量是3432.114599999999 g/mol

分子质量是3211.9924 g/mol

分子质量是2036.4180000000001 g/mol

分子质量是4276.265700000001 g/mol

分子质量是2546.9703999999997 g/mol

分子质量是1881.2

In [9]:
# 1. 显示所有列 (不省略中间的列)
pd.set_option('display.max_columns', None)

# 2. 显示所有行 (注意：如果数据有几万行，请谨慎使用，或改用 .head(100))
pd.set_option('display.max_rows', None)

# 3. 显示单元格内所有字符 (防止抗菌肽序列 Sequence 被截断)
pd.set_option('display.max_colwidth', None)

# 4. 确保输出宽度不换行
pd.set_option('display.width', 1000)
print(long_df["Raw_Species"].value_counts())

Raw_Species
MIC (E. coli)                             1212
MIC (S. aureus)                           1105
MIC (E. coli ATCC 25922)                   637
MIC (C. albicans)                          560
MIC (P. aeruginosa)                        513
MIC (B. subtilis)                          406
MIC (S. aureus ATCC 25923)                 401
MIC (S. epidermidis)                       299
MIC (P. aeruginosa ATCC 27853)             291
MIC (M. luteus)                            241
MIC (K. pneumoniae)                        238
MIC (A. baumannii)                         181
MIC (E. faecalis)                          179
MIC (E. faecium)                           156
MIC (S. aureus ATCC 29213)                 152
MIC (B. cereus)                            143
MIC (E. coli ATCC25922)                    142
MIC (L. monocytogenes)                     137
MIC (S. typhimurium)                       128
MIC (P. aeruginosa PAO1)                   128
MIC (S. enterica)                          127
M

In [28]:
import json
import time
with open("../AMP_COMMON_SPECIES.json", "r") as f:
    AMP_COMMON_SPECIES = json.load(f)

from Bio import Entrez
Entrez.email = "luyeqing21@mail.ustc.edu.cn"
def get_tax_id(species_name: str) -> str:
    term = f"{species_name}[Scientific Name]"
    handle = Entrez.esearch(db="taxonomy", term=term)
    record = Entrez.read(handle)

    if not record["IdList"]:
        print("No taxonomy ID found for the given species.")
        return None
    tax_id = record["IdList"][0]
    return str(tax_id)

SPECIES_TAX_ID = {}
tax_in_none = []
for species_name in AMP_COMMON_SPECIES.values():
    SPECIES_TAX_ID[species_name] = get_tax_id(species_name)
    time.sleep(0.5)  # 为了避免过于频繁的请求，添加延时
    if SPECIES_TAX_ID[species_name] is None:
        tax_in_none.append(species_name)

print(f"以下物种未找到对应的Taxonomy ID：{tax_in_none}")

with open("../SPECIES_TAX_ID.json", "w") as f:
    json.dump(SPECIES_TAX_ID, f, indent=4)


No taxonomy ID found for the given species.
No taxonomy ID found for the given species.
No taxonomy ID found for the given species.
No taxonomy ID found for the given species.
No taxonomy ID found for the given species.
No taxonomy ID found for the given species.
No taxonomy ID found for the given species.
以下物种未找到对应的Taxonomy ID：['Propionibacterium acnes', 'Candida glabrata', 'Lactobacillus salivarius', 'Listeria iavanovii', 'Candida krusei', 'Salmonella typhimurium', 'Salmonella enteritidis']


In [18]:
# 统一一下物种名称第一个是species，第二个是genes， 跟json文件对应起来

from rapidfuzz.distance import Levenshtein

with open("../POSSIBLE_GENES.json", "r") as f:
    POSSIBLE_GENES = json.load(f)

with open("../POSSIBLE_SPECIES.json", "r") as f:
    POSSIBLE_SPECIES = json.load(f)

with open("../SPECIES_TAX_ID.json", "r") as f:
    SPECIES_TAX_ID = json.load(f)

with open("../AMP_COMMON_SPECIES.json", "r") as f:
    AMP_COMMON_SPECIES = json.load(f)

def correct_species_with_distance(
    species_initial,
    genus_raw, 
    max_distance=2
):
    """
    raw_name: e.g. 'S. aureu'
    max_distance: 建议 1 或 2（非常重要）
    """

    best_match = None
    best_distance = None
    genus_list = POSSIBLE_GENES[species_initial]
    for genus in genus_list:
        dist = Levenshtein.distance(genus_raw, genus)

        if best_distance is None or dist < best_distance:
            best_distance = dist
            best_match = genus

    if best_distance is not None and best_distance <= max_distance:
        corrected = f"{species_initial}. {best_match}"
        return corrected, best_distance, "corrected"
    raw_name = f"{species_initial}. {genus_raw}"
    return raw_name, best_distance, "unchanged"

def NCBI_search(species:str, genus:str)->Tuple[str,str]:
    term = f"{species} {genus}[Scientific Name]"
    handle = Entrez.esearch(db="taxonomy", term=term)
    record = Entrez.read(handle)

    if not record["IdList"]:
        print(f"No taxonomy ID found for the given species:{species} {genus}.")
        return None, None
    tax_id = record["IdList"][0]
    summary = Entrez.efetch(db="taxonomy", id=tax_id, retmode="xml")
    data = Entrez.read(summary)

    sci_name = data[0]["ScientificName"]
    return tax_id, sci_name

def abbr2sci_name(abbr_name: str) -> Tuple[str, str]:
    abbr_name_set = AMP_COMMON_SPECIES.keys()
    # 缩写可以直接在本地找到
    if abbr_name in abbr_name_set:
        sci_name = AMP_COMMON_SPECIES[abbr_name]
        tax_id = SPECIES_TAX_ID[sci_name]  
        return tax_id, sci_name
    # 如果不在预定义的缩写中，先检查是否种名拼写有问题
    names = abbr_name.split()
    initial = names[0][0]
    genus = names[1]
    corrected_species, distance, status = correct_species_with_distance(initial, genus)
    if status == "corrected":
        sci_name = AMP_COMMON_SPECIES[corrected_species]
        tax_id = SPECIES_TAX_ID[sci_name]  
        return tax_id, sci_name
    
    # 最后尝试在POSSIBLE_SPECIES.json里面使用首字母组合，使用NCBI API搜索
    candidate_species = POSSIBLE_SPECIES[initial]
    for sp in candidate_species:
        tax_id, sci_name = NCBI_search(sp, genus)
        if tax_id:
            return tax_id, sci_name
        time.sleep(0.5)  # 为了避免过于频繁的请求，添加延时

    return None, None  # 如果不匹配，返回原始名称

In [21]:
name_df = pd.DataFrame()
# 1. 使用 .copy() 明确创建一个独立的对象，解决 SettingWithCopyWarning
name_df = long_df["Raw_Species"].drop_duplicates().to_frame(name="raw")
name_df = name_df.iloc[9:10].copy() 

# 2. 使用 apply 结合 pd.Series 批量更新，解决结果为 NaN 的问题
# 注意：result_type='expand' 可以将返回的元组拆分到三列
print("正在提取物种信息...")
name_df[["Tax_ID", "Target_Species", "Strain"]] = name_df["raw"].apply(
    lambda x: pd.Series(extract_bacterium_and_strain(x))
)

print(name_df)

正在提取物种信息...
                          raw Tax_ID           Target_Species Strain
57559  MIC (A. baumannii AB1)    470  Acinetobacter baumannii    AB1


In [22]:
long_df.head(3)

,APD_ID,Sequence,Length,Activity,Raw_Species,MIC_Value,MIC_Value(ug/ml)
3090,AP03328,TLKIIVKVVKYLV,13,"Anti-Gram+ & Gram-, Anti-MRSA, Hemolytic",MIC (A. acidoterrestris ATCC 49025),0.5 ug/ml),0.500000
8503,AP02677,FWQKMSFA,8,Anti-Gram+ & Gram-,MIC (A. acidoterrestris),16 uM),16.707608
9152,AP03328,TLKIIVKVVKYLV,13,"Anti-Gram+ & Gram-, Anti-MRSA, Hemolytic",MIC (A. acidoterrestris),0.5 ug/ml),0.500000


In [23]:
long_df = long_df.merge(name_df, left_on="Raw_Species",right_on="raw", how="left")
long_df.head(3)

,APD_ID,Sequence,Length,Activity,Raw_Species,MIC_Value,MIC_Value(ug/ml),raw,Tax_ID,Target_Species,Strain
0,AP03328,TLKIIVKVVKYLV,13,"Anti-Gram+ & Gram-, Anti-MRSA, Hemolytic",MIC (A. acidoterrestris ATCC 49025),0.5 ug/ml),0.500000,NaN,NaN,NaN,NaN
1,AP02677,FWQKMSFA,8,Anti-Gram+ & Gram-,MIC (A. acidoterrestris),16 uM),16.707608,NaN,NaN,NaN,NaN
2,AP03328,TLKIIVKVVKYLV,13,"Anti-Gram+ & Gram-, Anti-MRSA, Hemolytic",MIC (A. acidoterrestris),0.5 ug/ml),0.500000,NaN,NaN,NaN,NaN


In [68]:
long_df["Target_Species"] = long_df["Target_Species_y"]

In [69]:
long_df = long_df[["APD_ID", "Sequence", "Length", "Activity", "MIC_Value(ug/ml)", "Tax_ID", "Target_Species", "Strain"]]
long_df.to_csv("APD6_final_processed.csv", index=False)

In [13]:
# 把raw_species列中的变成 target_species，strain_ID 
from typing import Tuple
def extract_bacterium_and_strain(raw_text:str)->Tuple[str,str,str]:
    # 返回(tax_id : 562, species : Staphylococcus aureus, strain : ATCC 25922)
    match = re.search(r'MIC\s*\((.*)\)', str(raw_text))
    if not match:
        return None, None, None
    full_content = match.group(1).strip() # 例如 "A. baumanii CI 2675"
    parts = full_content.split()
    if len(parts) > 2:
        species_part = " ".join(parts[:2]) # "A. baumanii"
        strain_part = " ".join(parts[2:])   # "CI 2675"
    else:
        species_part = full_content
        strain_part = None

    tax_id, species = abbr2sci_name(species_part)
    if not tax_id:
        print(f"无法为物种名称找到Taxonomy ID: {species_part}\n")
        
    return tax_id, species, strain_part

In [50]:
print(extract_bacterium_and_strain("MIC (A. baumannii AB1)"))

('470', 'Acinetobacter baumannii', 'AB1')


In [ ]:
df = pd.read_csv("final_APD6_MIC_processed.csv", dtype={'Tax_ID':str})
df["Raw_Species"] = df["Raw_Species"].apply(lambda x: x[x.find('(')+1:x.find(')')] )

df['Raw_Species'].value_counts()

Raw_Species
E. coli                   1212
S. aureus                 1105
E. coli ATCC 25922         637
C. albicans                560
P. aeruginosa              513
                          ... 
A. ochraceus CGMCC 3         1
A. parasiticus               1
A. sydowii EF239             1
A. sydowii MEF 239           1
S. typhimurium CGMCC 1       1
Name: count, Length: 1163, dtype: int64

In [6]:
df.to_csv("final_APD6_MIC_processed.csv")